# TFG lechugas · v0 en Colab (GPU)

Este notebook **solo orquesta**: monta Drive, deja el repositorio en un commit concreto, instala las dependencias y lanza el pipeline (`python -m src.pipeline.run`) con el detector OWL-ViT sobre las fotos de prueba. Toda la lógica está en `src/`.

**Antes de empezar**
1. Entorno de ejecución → Cambiar tipo de entorno → GPU T4.
2. Sube las fotos de prueba a la carpeta de Drive que indica `IMAGENES` (por defecto `MyDrive/tfg-lechugas/data/v0_test`). Las fotos no van al repositorio.
3. En la celda de parámetros pon en `COMMIT` el commit exacto que quieras medir: así la medición se puede reproducir.

**Qué produce**
- `/content/runs/_descarga_owlvit`: primera pasada, solo para bajar el modelo y calentar la caché. No se copia ni se registra.
- `/content/runs/v0_colab_owlvit`: la medición válida (5 repeticiones tras 1 de calentamiento). Se copia a `MyDrive/tfg-lechugas/runs/v0_colab_owlvit`.

In [ ]:
# a) Parámetros. Es lo único que se edita antes de ejecutar.
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/diegocalvofm-ux/TFG-VisionArtificial.git"
COMMIT = "main"  # rama, etiqueta o hash de commit; para reproducir una medición, un hash
CARPETA_DRIVE = "/content/drive/MyDrive/tfg-lechugas"
IMAGENES = f"{CARPETA_DRIVE}/data/v0_test"  # fotos de prueba, fuera del repositorio

# Derivados: no hace falta tocarlos. Las rutas no deben llevar espacios.
REPO = "/content/TFG-VisionArtificial"
RUNS = "/content/runs"
PYTHON = sys.executable


def ejecutar(comando, cwd=None, comprobar=True):
    """Lanza un comando, enseña su salida según va saliendo y devuelve su código de salida.

    En lenguaje llano: es como escribir el comando en una terminal. Si el comando falla
    (código distinto de 0) lanza un error y el notebook se para en esa celda, en vez de seguir
    con un entorno a medias. Con comprobar=False solo devuelve el código.
    """
    print(f"$ {comando}", flush=True)
    entorno = {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8"}
    proceso = subprocess.Popen(
        shlex.split(comando),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=entorno,
    )
    for linea in proceso.stdout:
        print(linea, end="", flush=True)
    codigo = proceso.wait()
    if comprobar and codigo != 0:
        raise RuntimeError(f"El comando terminó con código {codigo}: {comando}")
    return codigo


In [ ]:
# b) Drive. Monta tu Drive y comprueba que las fotos están donde se espera.
import os

from google.colab import drive

drive.mount("/content/drive")

EXTENSIONES = (".jpg", ".jpeg", ".png", ".webp", ".bmp")  # las mismas que lee el pipeline
if not os.path.isdir(IMAGENES):
    raise FileNotFoundError(
        f"No existe la carpeta de fotos: {IMAGENES}. Sube las fotos de prueba a esa carpeta "
        "de Drive (o corrige IMAGENES en la celda de parámetros) y vuelve a ejecutar."
    )
fotos = [
    nombre
    for nombre in sorted(os.listdir(IMAGENES))
    if os.path.isfile(os.path.join(IMAGENES, nombre)) and nombre.lower().endswith(EXTENSIONES)
]
if not fotos:
    raise FileNotFoundError(
        f"No hay fotos ({', '.join(EXTENSIONES)}) en {IMAGENES}. Sube las fotos a esa carpeta."
    )
print(f"{len(fotos)} fotos en {IMAGENES}")


In [ ]:
# c) Repositorio. Lo clona (o lo actualiza) y lo deja en el commit pedido.
import os

if os.path.isdir(os.path.join(REPO, ".git")):
    ejecutar("git fetch --all --tags", cwd=REPO)
else:
    ejecutar(f"git clone {REPO_URL} {REPO}")
ejecutar(f"git checkout {COMMIT}", cwd=REPO)
# Si COMMIT es una rama se pone al día; con un hash o una etiqueta no hay nada que actualizar.
if ejecutar("git symbolic-ref -q HEAD", cwd=REPO, comprobar=False) == 0:
    ejecutar("git pull --ff-only", cwd=REPO)
os.chdir(REPO)
ejecutar("git rev-parse --short HEAD")  # el commit exacto que se va a medir: apúntalo


In [ ]:
# d) Dependencias. Colab ya trae torch y torchvision, por eso no se instalan.
ejecutar(f"{PYTHON} -m pip install -q -r requirements.txt -r requirements-detector.txt", cwd=REPO)


In [ ]:
# e) Entorno. Comprueba la GPU y deja una línea lista para copiar al registrar el experimento.
import subprocess
import sys

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU: Entorno de ejecución -> Cambiar tipo de entorno -> GPU T4")

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True
).strip()
print(
    "REGISTRO: "
    f"commit={commit} | GPU={torch.cuda.get_device_name(0)} | "
    f"transformers={transformers.__version__} | Python={sys.version.split()[0]} | "
    f"PyTorch={torch.__version__}"
)


In [ ]:
# f) Primera ejecución: descarga el modelo OWL-ViT y calienta la caché. Una sola pasada, sin
# calentamiento. Sus resultados no se copian ni se registran: sus tiempos no valen.
ejecutar(
    f"{PYTHON} -m src.pipeline.run --input {IMAGENES} --out {RUNS}/_descarga_owlvit"
    " --detector owlvit --repeticiones 1 --calentamiento 0",
    cwd=REPO,
)


In [ ]:
# g) Medición válida: con el modelo ya descargado, usa los valores por defecto del pipeline
# (5 repeticiones tras 1 de calentamiento) y escribe en una carpeta distinta.
ejecutar(
    f"{PYTHON} -m src.pipeline.run --input {IMAGENES} --out {RUNS}/v0_colab_owlvit"
    " --detector owlvit",
    cwd=REPO,
)


In [ ]:
# h) Copia la medición válida a Drive (para no perderla al cerrarse la sesión de Colab).
import os
import shutil

ORIGEN = f"{RUNS}/v0_colab_owlvit"
DESTINO = f"{CARPETA_DRIVE}/runs/v0_colab_owlvit"
shutil.copytree(ORIGEN, DESTINO, dirs_exist_ok=True)  # repetirla sobrescribe, no falla
for nombre in sorted(os.listdir(DESTINO)):
    kilobytes = os.path.getsize(os.path.join(DESTINO, nombre)) / 1024
    print(f"{nombre}  ({kilobytes:.1f} KB)")
print(f"Copiado en {DESTINO}")


## Después

1. Descarga la carpeta `MyDrive/tfg-lechugas/runs/v0_colab_owlvit` de Drive a `runs/v0_colab_owlvit` del repositorio local (la carpeta `runs/` no se sube a git).
2. Copia la línea que empieza por `REGISTRO:` de la salida de la celda e): lleva el commit corto, el modelo de GPU y las versiones de `transformers`, Python y PyTorch.
3. Registra la ejecución en Obsidian con Claude Code y pega esa línea en las observaciones:

   `/registrar-experimento exp001 runs/v0_colab_owlvit`

   Las observaciones del registro deben incluir esa línea: el JSON solo trae `device=cuda` y no guarda el modelo de GPU ni la versión de `transformers`, así que, si no la copias, esos datos se pierden al cerrar la sesión de Colab.

Los tiempos de esta medición son de la GPU de Colab; los de `runs/v0_owlvit` son de la CPU del portátil.